# 🧬 Pipeline Unificado de Splicing con CNN 1D
## Notebook para Google Colab

**Autores:** César Alexander Martínez Pérez | Guillermo Daniel Zaragoza Castro  
**Institución:** CUCEI, Universidad de Guadalajara  
**Fecha:** Mayo 2026

---

### Contenido:
1. Configuración
2. Carga de datos
3. Entrenamiento
4. Evaluación
5. Predicciones


## 1. Configuración

In [ ]:
import os, sys, random, numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report, confusion_matrix
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print(f'PyTorch: {torch.__version__}')

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = '/content/drive/MyDrive'
except: 
    DRIVE_ROOT = None

## 2. Arquitectura

In [ ]:
class SplicingCNN1D(nn.Module):
    def __init__(self, filtros=[64,128,256], kernels=[15,11,9], dropout=0.4):
        super().__init__()
        self.conv1 = nn.Conv1d(4, filtros[0], kernels[0], padding=kernels[0]//2)
        self.bn1 = nn.BatchNorm1d(filtros[0])
        self.pool1 = nn.MaxPool1d(4)
        self.conv2 = nn.Conv1d(filtros[0], filtros[1], kernels[1], padding=kernels[1]//2)
        self.bn2 = nn.BatchNorm1d(filtros[1])
        self.pool2 = nn.MaxPool1d(4)
        self.conv3 = nn.Conv1d(filtros[1], filtros[2], kernels[2], padding=kernels[2]//2)
        self.bn3 = nn.BatchNorm1d(filtros[2])
        self.pool3 = nn.MaxPool1d(3)
        self.fc1 = nn.Linear(filtros[2]*4, 256)
        self.fc2 = nn.Linear(256, 1)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x):
        x = x.permute(0,2,1)
        x = self.pool1(self.dropout(F.relu(self.bn1(self.conv1(x)))))
        x = self.pool2(self.dropout(F.relu(self.bn2(self.conv2(x)))))
        x = self.pool3(self.dropout(F.relu(self.bn3(self.conv3(x)))))
        x = x.view(x.size(0), -1)
        return torch.sigmoid(self.fc2(self.dropout(F.relu(self.fc1(x)))))

## 3. Dataset

In [ ]:
def codificar_one_hot(secuencia, max_length=200):
    mapeo = {'A': [1,0,0,0], 'C': [0,1,0,0], 'G': [0,0,1,0], 'T': [0,0,0,1], 'N': [0,0,0,0]}
    matriz = np.zeros((max_length, 4), dtype=np.float32)
    for i, nuc in enumerate(secuencia.upper()[:max_length]):
        if nuc in mapeo: matriz[i] = mapeo[nuc]
    return torch.from_numpy(matriz)

class SplicingDataset(Dataset):
    def __init__(self, csv_path, max_length=200):
        self.df = pd.read_csv(csv_path, usecols=['sequence', 'label'])
        self.max_length = max_length
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        return codificar_one_hot(row['sequence'], self.max_length), torch.tensor(row['label'], dtype=torch.float32)

## 4. Entrenamiento

In [ ]:
def entrenar_epoch(modelo, loader, criterion, optimizer, device):
    modelo.train()
    total_loss, total_acc, n = 0, 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        out = modelo(x).squeeze()
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        total_acc += ((out >= 0.5).float() == y).sum().item()
        n += len(y)
    return total_loss / len(loader), total_acc / n * 100

## 5. Entrenamiento Completo


In [ ]:
def entrenar_modelo_completo(train_path, val_path, num_epochs=50, lr=0.001, batch_size=32, save_path=None):
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f'Dispositivo: {DEVICE}')
    
    # Cargar datasets
    train_dataset = SplicingDataset(train_path)
    val_dataset = SplicingDataset(val_path)
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size)
    
    # Crear modelo
    modelo = SplicingCNN1D()
    modelo = modelo.to(DEVICE)
    
    criterion = nn.BCELoss()
    optimizer = torch.optim.Adam(modelo.parameters(), lr=lr)
    
    historial = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    best_val_loss = float('inf')
    
    print(f'\nIniciando entrenamiento ({num_epochs} épocas)...')
    for epoch in range(1, num_epochs + 1):
        train_loss, train_acc = entrenar_epoch(modelo, train_loader, criterion, optimizer, DEVICE)
        val_loss, val_acc = validar_epoch(modelo, val_loader, criterion, DEVICE)
        
        historial['train_loss'].append(train_loss)
        historial['val_loss'].append(val_loss)
        historial['train_acc'].append(train_acc)
        historial['val_acc'].append(val_acc)
        
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            if save_path:
                torch.save({'model_state_dict': modelo.state_dict(), 'best_val_loss': best_val_loss, 'historial': historial}, save_path)
        
        if epoch % 5 == 0 or epoch == 1:
            print(f'Época {epoch:3d} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%')
    
    print(f'\n✓ Entrenamiento completado. Mejor Val Loss: {best_val_loss:.4f}')
    if save_path:
        print(f'✓ Modelo guardado: {save_path}')
    
    return modelo, historial

def validar_epoch(modelo, loader, criterion, device):
    modelo.eval()
    total_loss, total_acc, n = 0, 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = modelo(x).squeeze()
            loss = criterion(out, y)
            total_loss += loss.item()
            total_acc += ((out >= 0.5).float() == y).sum().item()
            n += len(y)
    return total_loss / len(loader), total_acc / n * 100

## 6. Métricas y Visualización


In [ ]:
def plotear_curvas(historial, save_path=None):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Loss
    ax1.plot(historial['train_loss'], 'b-', label='Train Loss', linewidth=2)
    ax1.plot(historial['val_loss'], 'r-', label='Val Loss', linewidth=2)
    ax1.set_xlabel('Época')
    ax1.set_ylabel('Loss (BCE)')
    ax1.set_title('Curva de Pérdida')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Accuracy
    ax2.plot(historial['train_acc'], 'b-', label='Train Accuracy', linewidth=2)
    ax2.plot(historial['val_acc'], 'r-', label='Val Accuracy', linewidth=2)
    ax2.set_xlabel('Época')
    ax2.set_ylabel('Accuracy (%)')
    ax2.set_title('Curva de Precisión')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    ax2.set_ylim([0, 100])
    
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150)
        print(f'✓ Gráficas guardadas: {save_path}')
    plt.show()

def mostrar_metricas(historial):
    import numpy as np
    print('RESUMEN DE MÉTRICAS')
    print('='*50)
    print(f'Épocas: {len(historial["train_loss"])}')
    print(f'Mejor Val Loss: {np.min(historial["val_loss"]):.4f}')
    print(f'Época óptima: {np.argmin(historial["val_loss"])+1}')
    print(f'\nÚltima época:')
    print(f'  Train Loss: {historial["train_loss"][-1]:.4f}')
    print(f'  Val Loss: {historial["val_loss"][-1]:.4f}')
    print(f'  Train Acc: {historial["train_acc"][-1]:.2f}%')
    print(f'  Val Acc: {historial["val_acc"][-1]:.2f}%')
    print('='*50)

## 7. Predicciones y Evaluación


In [ ]:
def predecir_secuencia(modelo, secuencia, threshold=0.5):
    """Predice si una secuencia es sitio verdadero o señuelo."""
    modelo.eval()
    DEVICE = next(modelo.parameters()).device
    x = codificar_one_hot(secuencia, 200).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        prob = modelo(x).item()
    clase = 1 if prob >= threshold else 0
    etiqueta = 'Sitio Verdadero' if clase == 1 else 'Señuelo'
    return prob, clase, etiqueta

def predecir_lote(modelo, secuencias, threshold=0.5):
    """Predice múltiples secuencias."""
    modelo.eval()
    resultados = []
    for seq in secuencias:
        prob, clase, etiqueta = predecir_secuencia(modelo, seq, threshold)
        resultados.append((prob, clase, etiqueta))
    return resultados

## 8. Ejemplo de Uso


In [ ]:
# Ejemplo 1: Entrenar modelo
if DRIVE_ROOT:
    modelo, historial = entrenar_modelo_completo(
        train_path=os.path.join(DRIVE_ROOT, 'dataset_entrenamiento.csv'),
        val_path=os.path.join(DRIVE_ROOT, 'dataset_prueba.csv'),
        num_epochs=50,
        save_path=os.path.join(DRIVE_ROOT, 'modelo_splicing_best.pth')
    )
    
    # Visualizar métricas
    mostrar_metricas(historial)
    plotear_curvas(historial, save_path=os.path.join(DRIVE_ROOT, 'curvas_entrenamiento.png'))
else:
    print('Google Drive no disponible. Sube tus datasets a Drive para entrenar.')

In [ ]:
# Ejemplo 2: Predecir con modelo cargado
secuencias_ejemplo = ['ACGT' * 50, 'TGCA' * 50, 'AAAA' * 50]
resultados = predecir_lote(modelo, secuencias_ejemplo)

for i, (prob, clase, etiqueta) in enumerate(resultados):
    print(f'{i+1}. {etiqueta} (prob: {prob:.4f})')